# STAIR-v3.1: ClipFuse-Consensus — Prior-Preserving Fusion + Cross-Modal Consensus Boosting

**Key Innovations:**
- **Prior-Preserving Weighting:** Preserves domain structural prior (Text k=5, Visual k=1 → 83.3% Text : 16.7% Visual).
- **Cross-Modal Consensus Boosting (α=0.5):** Overlapping edges in BOTH Text and Visual kNN receive ×1.5 weight multiplier.

| Parameter | Value |
|---|---|
| Dataset | Amazon2014Baby + Amazon2014Sports |
| Epochs | 500 |
| Neighbors | 5-1 (83.3% Text : 16.7% Visual) |
| Consensus Boost (α) | 0.5 (×1.5 multiplier for overlapping edges) |
| δ (floor) | 0.3 |

In [ ]:
# Cell 1: Setup & Install
import os, shutil

os.chdir('/kaggle/working')

repo = 'STAIR-Enhanced'
if os.path.exists(repo):
    shutil.rmtree(repo)
os.system('git clone https://github.com/ThanhChuong12/STAIR-Enhanced.git')

os.system('pip install nvidia-ml-py -q')
os.system('pip install torchdata==0.6.1 --no-deps -q')
os.system('pip install freerec==0.9.7 -q')
os.system('pip install torch_geometric -q')
os.system('pip install prettytable -q')

import torch, platform, freerec
print('Python  :', platform.python_version())
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('freerec :', freerec.__version__)
print('Environment ready')


In [ ]:
# Cell 2: Copy dataset files
import os, shutil

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)

def copy_dataset(keywords, full_name):
    dest = os.path.join(DATA_ROOT, full_name)
    os.makedirs(dest, exist_ok=True)
    copied = []
    for root, _, files in os.walk('/kaggle/input'):
        if any(kw.lower() in root.lower() for kw in keywords):
            for f in files:
                if f.endswith(('.npy', '.pkl', '.txt', '.inter', '.item')):
                    shutil.copy(os.path.join(root, f), os.path.join(dest, f))
                    copied.append(f)
    print(f'[{full_name}] {len(copied)} files copied')

copy_dataset(['baby', 'amazon2014baby'],     'Amazon2014Baby_550_MMRec')
copy_dataset(['sports', 'amazon2014sports'], 'Amazon2014Sports_550_MMRec')
print('Data ready at', DATA_ROOT)


In [ ]:
# Cell 3: ClipFuse-v3.1 Structural Diagnostics
import os, sys, pickle, warnings, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, '/kaggle/working/STAIR-Enhanced')
warnings.filterwarnings('ignore')

DATA_ROOT = '/kaggle/data'
DATASETS  = ['Amazon2014Baby_550_MMRec', 'Amazon2014Sports_550_MMRec']
MFILES    = ['textual_modality.pkl', 'visual_modality.pkl']

def load_feat(ds, mf):
    with open(os.path.join(DATA_ROOT, ds, mf), 'rb') as f:
        feat = pickle.load(f)
    if not isinstance(feat, torch.Tensor):
        feat = torch.tensor(feat, dtype=torch.float32)
    return feat.float()

def build_knn_sims(feat, k):
    feat_n = F.normalize(feat, p=2, dim=-1)
    sim = feat_n @ feat_n.t()
    sim.fill_diagonal_(-10.)
    topk_vals, _ = sim.topk(k=k, dim=-1)
    return topk_vals.clamp(min=0.).mean().item()

K_TEXT, K_VIS = 5, 1
ALPHA_CONSENSUS = 0.5

for ds in DATASETS:
    ds_name = 'Baby' if 'Baby' in ds else 'Sports'
    try:
        ft = load_feat(ds, MFILES[0])
        fv = load_feat(ds, MFILES[1])
        mean_sim_t = build_knn_sims(ft, K_TEXT)
        mean_sim_v = build_knn_sims(fv, K_VIS)
        disc_t = max(0., 1. - mean_sim_t)
        disc_v = max(0., 1. - mean_sim_v)
        exp_v = math.exp(disc_v); exp_t = math.exp(disc_t)
        c_v = max(0.3, min(0.7, exp_v / (exp_v + exp_t)))
        c_t = 1.0 - c_v
        w_t = K_TEXT * c_t; w_v = K_VIS * c_v
        pct_t = w_t / (w_t + w_v) * 100; pct_v = 100 - pct_t
        print(f'[{ds_name}] mean_sim: t={mean_sim_t:.4f}, v={mean_sim_v:.4f}')
        print(f'  confidence: c_t={c_t:.4f}, c_v={c_v:.4f}')
        print(f'  weight pool: Text={w_t:.3f} ({pct_t:.1f}%), Visual={w_v:.3f} ({pct_v:.1f}%)')
    except FileNotFoundError:
        print(f'[WARN] Data not found for {ds_name}. Run Cell 2 first.')

print('Diagnostics done.')


In [ ]:
# Cell 4: Train STAIR-v3.1 on Baby
import os, subprocess, time, shutil

os.chdir('/kaggle/working/STAIR-Enhanced')

log_path_baby = '/kaggle/working/log_stair_v3_baby.txt'

print('Training STAIR-v3.1 (ClipFuse-Consensus) on Baby...')
t0 = time.time()
result = subprocess.run(
    ['python', 'main_v3.py',
     '--root', '/kaggle/data',
     '--dataset', 'Amazon2014Baby_550_MMRec',
     '--epochs', '500', '--batch-size', '1024',
     '--embedding-dim', '64', '--num-layers', '3',
     '--num-neighbors', '5-1', '--conf-delta', '0.3',
     '--conf-temp', '1.0', '--alpha-consensus', '0.5',
     '--optimizer', 'adamwsevo', '--lr', '1e-3',
     '--weight-decay', '0.1', '--seed', '1'],
    stdout=open(log_path_baby, 'w'),
    stderr=subprocess.STDOUT, text=True
)
elapsed = (time.time() - t0) / 60
print(f'Done in {elapsed:.1f} min | rc={result.returncode}')

with open(log_path_baby) as f:
    lines = f.readlines()
if result.returncode != 0:
    print('\n[ERROR] Last 30 lines:')
    print(''.join(lines[-30:]))
else:
    print('\nLast 5 lines:')
    print(''.join(lines[-5:]))

os.makedirs('/kaggle/working/STAIR-Enhanced/logs', exist_ok=True)
shutil.copy(log_path_baby, '/kaggle/working/STAIR-Enhanced/logs/log_stair_v3_baby.txt')


In [ ]:
# Cell 5: Train STAIR-v3.1 on Sports
import os, subprocess, time, shutil

os.chdir('/kaggle/working/STAIR-Enhanced')

log_path_sports = '/kaggle/working/log_stair_v3_sports.txt'

print('Training STAIR-v3.1 (ClipFuse-Consensus) on Sports...')
t0 = time.time()
result = subprocess.run(
    ['python', 'main_v3.py',
     '--root', '/kaggle/data',
     '--dataset', 'Amazon2014Sports_550_MMRec',
     '--epochs', '500', '--batch-size', '1024',
     '--embedding-dim', '64', '--num-layers', '3',
     '--num-neighbors', '5-1', '--conf-delta', '0.3',
     '--conf-temp', '1.0', '--alpha-consensus', '0.5',
     '--optimizer', 'adamwsevo', '--lr', '1e-3',
     '--weight-decay', '0.1', '--seed', '1'],
    stdout=open(log_path_sports, 'w'),
    stderr=subprocess.STDOUT, text=True
)
elapsed = (time.time() - t0) / 60
print(f'Done in {elapsed:.1f} min | rc={result.returncode}')

with open(log_path_sports) as f:
    lines = f.readlines()
if result.returncode != 0:
    print('\n[ERROR] Last 30 lines:')
    print(''.join(lines[-30:]))
else:
    print('\nLast 5 lines:')
    print(''.join(lines[-5:]))

os.makedirs('/kaggle/working/STAIR-Enhanced/logs', exist_ok=True)
shutil.copy(log_path_sports, '/kaggle/working/STAIR-Enhanced/logs/log_stair_v3_sports.txt')


In [ ]:
# Cell 6: Parse Logs to Extract Validation Curves and Final TEST Metrics
import os, re
import numpy as np

def parse_stair_log(log_path):
    if not os.path.exists(log_path):
        return None
    
    losses, loss_eps = [], []
    val_eps, r10, r20, n10, n20 = [], [], [], [], []
    test_metrics = None
    
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        
    # Parse losses
    for m in re.finditer(r'TRAIN @Epoch:\s*(\d+).*?LOSS Avg:\s*([0-9.]+)', content):
        loss_eps.append(int(m.group(1)))
        losses.append(float(m.group(2)))
        
    # Parse validation (flatten lines to handle any splits)
    content_flat = content.replace('\n', ' ')
    for m in re.finditer(r'VALID @Epoch:\s*(\d+)\s*>>>\s*\|\|\s*RECALL@10 Avg:\s*([0-9.]+)\s*\|\|\s*RECALL@20 Avg:\s*([0-9.]+)\s*\|\|\s*NDCG@10 Avg:\s*([0-9.]+)\s*\|\|\s*NDCG@20 Avg:\s*([0-9.]+)', content_flat):
        val_eps.append(int(m.group(1)))
        r10.append(float(m.group(2)))
        r20.append(float(m.group(3)))
        n10.append(float(m.group(4)))
        n20.append(float(m.group(5)))
        
    # Parse final TEST metrics (after Load best model)
    test_match = re.search(r'Load best model @Epoch.*?TEST  @Epoch:\s*\d+\s*>>>\s*\|\|\s*RECALL@10 Avg:\s*([0-9.]+)\s*\|\|\s*RECALL@20 Avg:\s*([0-9.]+)\s*\|\|\s*NDCG@10 Avg:\s*([0-9.]+)\s*\|\|\s*NDCG@20 Avg:\s*([0-9.]+)', content_flat)
    if test_match:
        test_metrics = {
            'Recall@10': float(test_match.group(1)),
            'Recall@20': float(test_match.group(2)),
            'NDCG@10': float(test_match.group(3)),
            'NDCG@20': float(test_match.group(4))
        }
        
    return {
        'loss_eps': loss_eps, 'losses': losses,
        'val_eps': val_eps, 'r10': r10, 'r20': r20, 'n10': n10, 'n20': n20,
        'test_metrics': test_metrics
    }

baby_data = parse_stair_log('/kaggle/working/log_stair_v3_baby.txt') or parse_stair_log('logs/log_stair_v3_baby.txt')
sports_data = parse_stair_log('/kaggle/working/log_stair_v3_sports.txt') or parse_stair_log('logs/log_stair_v3_sports.txt')

if baby_data is None:
    print('[WARN] Baby log not found, using hardcoded test metrics.')
    baby_data = {'test_metrics': {'Recall@10': 0.0612, 'Recall@20': 0.0938, 'NDCG@10': 0.0329, 'NDCG@20': 0.0412}, 'loss_eps': [], 'losses': [], 'val_eps': [], 'r10': [], 'r20': [], 'n10': [], 'n20': []}
if sports_data is None:
    print('[WARN] Sports log not found, using hardcoded test metrics.')
    sports_data = {'test_metrics': {'Recall@10': 0.0740, 'Recall@20': 0.1124, 'NDCG@10': 0.0406, 'NDCG@20': 0.0506}, 'loss_eps': [], 'losses': [], 'val_eps': [], 'r10': [], 'r20': [], 'n10': [], 'n20': []}

test_baby_v3 = baby_data['test_metrics']
test_sports_v3 = sports_data['test_metrics']

print('=' * 70)
print('FINAL TEST METRICS (from "Load best model" checkpoint):')
print(f'  Baby  : {test_baby_v3}')
print(f'  Sports: {test_sports_v3}')
print('=' * 70)
print(f"Baby  val points: {len(baby_data['val_eps'])}, loss points: {len(baby_data['loss_eps'])}")
print(f"Sports val points: {len(sports_data['val_eps'])}, loss points: {len(sports_data['loss_eps'])}")


In [ ]:
# Cell 7: Learning Curves — actual per-epoch data from log
import numpy as np
import matplotlib.pyplot as plt

def smooth(values, w=10):
    if len(values) <= w: return values
    return np.convolve(values, np.ones(w)/w, mode='valid').tolist()

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('STAIR-v3.1 (ClipFuse-Consensus) — Learning Curves (Log Data)', fontsize=15, fontweight='bold')

datasets_info = [
    ('Baby', baby_data['loss_eps'], baby_data['losses'], baby_data['val_eps'], baby_data['r10'], baby_data['r20'], baby_data['n10'], baby_data['n20']),
    ('Sports', sports_data['loss_eps'], sports_data['losses'], sports_data['val_eps'], sports_data['r10'], sports_data['r20'], sports_data['n10'], sports_data['n20'])
]

for ci, (ds_name, loss_ep, loss_v, val_ep, r10, r20, n10, n20) in enumerate(datasets_info):
    if not val_ep:
        continue
    
    # Trim to match lengths safely
    m_len = min(len(val_ep), len(r10), len(r20), len(n10), len(n20))
    v_ep, r10_c, r20_c, n10_c, n20_c = val_ep[:m_len], r10[:m_len], r20[:m_len], n10[:m_len], n20[:m_len]

    # Row 0: Loss
    ax = axes[0][ci]
    if loss_v:
        s = smooth(loss_v, w=10)
        offset = (10 - 1) // 2
        ax.plot(loss_ep, loss_v, color='#E8C55A', alpha=0.3, linewidth=0.8, label='Raw Loss')
        ax.plot(loss_ep[offset: offset + len(s)], s, color='#E8734A', linewidth=2, label='Smoothed (w=10)')
    ax.set_title(f'{ds_name} — BPR Training Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Row 1: Recall
    ax = axes[1][ci]
    ax.plot(v_ep, r10_c, color='#2ECC71', linewidth=2.0, marker='o', markersize=2, label='Recall@10')
    ax.plot(v_ep, r20_c, color='#1A7A3F', linewidth=2.0, marker='s', markersize=2, label='Recall@20')
    ax.axhline(y=max(r10_c), color='#2ECC71', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axhline(y=max(r20_c), color='#1A7A3F', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.set_title(f'{ds_name} — Recall@K', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Recall'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f'Max R@20={max(r20_c):.4f}', xy=(v_ep[r20_c.index(max(r20_c))], max(r20_c)),
                fontsize=7, color='#1A7A3F',
                xytext=(10, -15), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='#1A7A3F', lw=0.8))

    # Row 2: NDCG
    ax = axes[2][ci]
    ax.plot(v_ep, n10_c, color='#3498DB', linewidth=2.0, marker='o', markersize=2, label='NDCG@10')
    ax.plot(v_ep, n20_c, color='#1A5276', linewidth=2.0, marker='s', markersize=2, label='NDCG@20')
    ax.axhline(y=max(n10_c), color='#3498DB', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axhline(y=max(n20_c), color='#1A5276', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.set_title(f'{ds_name} — NDCG@K', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('NDCG'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f'Max N@20={max(n20_c):.4f}', xy=(v_ep[n20_c.index(max(n20_c))], max(n20_c)),
                fontsize=7, color='#1A5276',
                xytext=(10, -15), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='#1A5276', lw=0.8))

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curves saved.')


In [ ]:
# Cell 8: Full Performance Comparison Table (Test Metrics)
from prettytable import PrettyTable

# BASELINE TEST METRICS
BASELINE = {
    'Baby':   {'Recall@10': 0.068560, 'Recall@20': 0.103420, 'NDCG@10': 0.036335, 'NDCG@20': 0.045143},
    'Sports': {'Recall@10': 0.074310, 'Recall@20': 0.111900, 'NDCG@10': 0.040200, 'NDCG@20': 0.050050},
}
# V1 GCL TEST METRICS
V1_GCL = {
    'Baby':   {'Recall@10': 0.069459, 'Recall@20': 0.104700, 'NDCG@10': 0.036535, 'NDCG@20': 0.045531},
    'Sports': {'Recall@10': 0.074541, 'Recall@20': 0.112394, 'NDCG@10': 0.040429, 'NDCG@20': 0.050400},
}
# V2 DYFUSE TEST METRICS
V2_DYFUSE = {
    'Baby':   {'Recall@10': 0.057800, 'Recall@20': 0.089800, 'NDCG@10': 0.030900, 'NDCG@20': 0.038800},
    'Sports': {'Recall@10': 0.066900, 'Recall@20': 0.100200, 'NDCG@10': 0.036600, 'NDCG@20': 0.044900},
}
# V3 TEST METRICS (Parsed directly from logs after 'Load best model')
V3_CLIPFUSE = {
    'Baby':   test_baby_v3,
    'Sports': test_sports_v3,
}

METRICS  = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']
DATA_MAP = [BASELINE, V1_GCL, V2_DYFUSE, V3_CLIPFUSE]

print('=' * 115)
print('TEST COMPARISON: Baseline  vs  v1 (GCL)  vs  v2 (DyFuse)  vs  v3 (ClipFuse-Consensus)')
print('=' * 115)

for ds in ['Baby', 'Sports']:
    t = PrettyTable()
    t.field_names = ['Metric', 'Baseline', 'v1 (GCL)', 'v2 (DyFuse)', 'v3 (ClipFuse)', 'v3 vs Base', 'v3 vs v1', 'v3 vs v2']
    for metric in METRICS:
        vals = [data[ds].get(metric, 0) for data in DATA_MAP]
        v3 = vals[3]
        t.add_row([metric,
            f'{vals[0]:.6f}', f'{vals[1]:.6f}', f'{vals[2]:.6f}', f'{vals[3]:.6f}',
            f'{(v3-vals[0])/(vals[0]+1e-9)*100:+.2f}%',
            f'{(v3-vals[1])/(vals[1]+1e-9)*100:+.2f}%',
            f'{(v3-vals[2])/(vals[2]+1e-9)*100:+.2f}%',
        ])
    print(f'\nDataset: {ds}'); print(t)
print('=' * 115)


In [ ]:
# Cell 9: Bar Chart Comparison
import numpy as np
import matplotlib.pyplot as plt

METRICS  = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']
COLORS   = ['#95A5A6', '#3498DB', '#E74C3C', '#2ECC71']
LABELS   = ['Baseline', 'v1 (GCL)', 'v2 (DyFuse)', 'v3 (ClipFuse)']
DATA_MAP = [BASELINE, V1_GCL, V2_DYFUSE, V3_CLIPFUSE]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model Comparison — Baseline vs v1 (GCL) vs v2 (DyFuse) vs v3 (ClipFuse)',
             fontsize=13, fontweight='bold')

for ri, ds in enumerate(['Baby', 'Sports']):
    for ci, metric in enumerate(METRICS):
        ax = axes[ri][ci]
        vals = [data[ds].get(metric, 0) for data in DATA_MAP]
        bars = ax.bar(LABELS, vals, color=COLORS, alpha=0.85, edgecolor='white', width=0.6)
        ax.set_title(f'{ds} — {metric}', fontweight='bold', fontsize=10)
        ax.set_ylim(0, max(vals) * 1.25)
        ax.tick_params(axis='x', rotation=25, labelsize=7)
        ax.grid(alpha=0.3, axis='y')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=8,
                    fontweight='bold' if val == max(vals) else 'normal')

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 10: Improvement Heatmap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

comparisons = [
    ('v1 vs Base', V1_GCL, BASELINE), ('v2 vs Base', V2_DYFUSE, BASELINE),
    ('v3 vs Base', V3_CLIPFUSE, BASELINE), ('v3 vs v1', V3_CLIPFUSE, V1_GCL),
    ('v3 vs v2', V3_CLIPFUSE, V2_DYFUSE),
]
rows = [f'{ds} — {m}' for ds in ['Baby', 'Sports'] for m in METRICS]
cols = [c[0] for c in comparisons]
data = np.zeros((len(rows), len(cols)))

for ci, (_, model, ref) in enumerate(comparisons):
    ri = 0
    for ds in ['Baby', 'Sports']:
        for metric in METRICS:
            m_val = model.get(ds, {}).get(metric, 0)
            r_val = ref.get(ds, {}).get(metric, 1e-9)
            data[ri, ci] = (m_val - r_val) / (r_val + 1e-9) * 100
            ri += 1

fig, ax = plt.subplots(figsize=(12, 8))
vmax = max(10, data.max() * 0.8)
vmin = min(-10, data.min() * 0.8)
im = ax.imshow(data, cmap='RdYlGn', aspect='auto',
               norm=mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax))
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=10)
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=9)
plt.colorbar(im, ax=ax, label='Relative Improvement (%)')
for i in range(len(rows)):
    for j in range(len(cols)):
        val = data[i, j]
        ax.text(j, i, f'{val:+.1f}%', ha='center', va='center', fontsize=9,
                fontweight='bold', color='white' if abs(val) > 8 else 'black')
ax.set_title('Relative Improvement (%) — TEST Metrics vs Baselines', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 11: Final Summary
import os, shutil

print('=' * 70)
print('STAIR-v3 ClipFuse — Final TEST Results (Parsed from logs)')
print('=' * 70)

for ds, best in [('Baby', test_baby_v3), ('Sports', test_sports_v3)]:
    base = BASELINE[ds]
    print(f'\n[{ds}]')
    for metric in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
        v3 = best.get(metric, 0)
        bl = base[metric]
        d  = (v3 - bl) / (bl + 1e-9) * 100
        flag = '✓ BETTER' if v3 >= bl else '✗ WORSE'
        print(f'  {metric:<12}: v3={v3:.6f}  baseline={bl:.6f}  delta={d:+.2f}%  {flag}')

print('\n' + '=' * 70)
output_files = [
    '/kaggle/working/stair_v3_learning_curves.png',
    '/kaggle/working/stair_v3_comparison.png',
    '/kaggle/working/stair_v3_heatmap.png',
]
for fp in output_files:
    if os.path.exists(fp):
        print(f'  [OK]   {os.path.basename(fp):<50} {os.path.getsize(fp)/1024:.1f} KB')
    else:
        print(f'  [MISS] {os.path.basename(fp)}')
